# 🧪 Sesión 33 · Servidores ASGI / WSGI en Producción (Uvicorn / Nginx)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Quién atiende a miles de usuarios a la vez?

---

## 🖥️ Así se verá el resultado final

```text
$ gunicorn -c gunicorn_conf.py app:app
[INFO] Starting gunicorn 21.2.0
[INFO] Listening at: http://0.0.0.0:8000
[INFO] Using worker: uvicorn.workers.UvicornWorker
[INFO] Booting worker with pid: 101
[INFO] Booting worker with pid: 102
[INFO] Booting worker with pid: 103

$ curl -I https://api.ejemplo.com/api/libros
HTTP/2 200
server: nginx
content-type: application/json

nginx.conf: proxy_pass -> 127.0.0.1:8000
gunicorn_conf.py: workers = 2 * cpu_count() + 1
start.sh: sin --reload
```

_En producción no se usa `uvicorn --reload`: consume memoria y reinicia el proceso en cada cambio._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Configurar workers de Gunicorn/Uvicorn.
- Escribir un proxy inverso con Nginx.
- Verificar que no se usa --reload en producción.

## ✅ Criterios de aceptación

El verificador comprobará estos criterios sobre los archivos que produzcas:

- [ ] Gunicorn usa la fórmula de workers
- [ ] Nginx hace proxy y limita el cuerpo
- [ ] Arranque de producción sin reload
- [ ] La fórmula de workers es la esperada

## 🧩 El reto

Escribe la configuración de Gunicorn, el proxy inverso de Nginx y el script de arranque para producción.

**Requisitos del contrato**

- gunicorn_conf.py calcula workers = 2 * CPU + 1.
- Usa uvicorn.workers.UvicornWorker como clase de worker.
- nginx.conf hace proxy_pass hacia la aplicación y activa gzip.
- Nginx escucha en el puerto 80 y limita el tamaño del cuerpo.
- start.sh arranca con gunicorn y no contiene --reload.

> **Pista conceptual:** La fórmula de workers no es un número mágico: equilibra CPU e I/O. Nginx absorbe la red; Gunicorn atiende la aplicación.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion33.md` con la arquitectura Nginx → Gunicorn → Uvicorn.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/main/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `gunicorn_conf.py`, `nginx.conf`, `start.sh`


### 📄 `gunicorn_conf.py`

Configuración de Gunicorn.

In [ ]:
%%writefile gunicorn_conf.py
import multiprocessing

# Define workers, worker_class, bind y timeout.


### 📄 `nginx.conf`

Proxy inverso.

In [ ]:
%%writefile nginx.conf
# Define el upstream, el server, el proxy_pass y gzip.


### 📄 `start.sh`

Script de arranque.

In [ ]:
%%writefile start.sh
#!/usr/bin/env bash
set -e
# Arranca la aplicación en modo producción.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("gunicorn_conf.py", regex=["workers\\s*=\\s*\\(?\\s*2\\s*\\*", "cpu_count|multiprocessing", "UvicornWorker"], descripcion="Gunicorn usa la fórmula de workers"))
resultados.append(revisar_archivo("nginx.conf", regex=["proxy_pass\\s+http://(127\\.0\\.0\\.1|localhost|api)", "gzip\\s+on", "client_max_body_size", "listen\\s+80"], descripcion="Nginx hace proxy y limita el cuerpo"))
resultados.append(revisar_archivo("start.sh", exigir=["gunicorn"], prohibir=["--reload"], descripcion="Arranque de producción sin reload"))
resumen(resultados)


In [ ]:
from verificar import revisar_python, resumen

resultados = []
resultados.append(revisar_python("import multiprocessing\nimport gunicorn_conf as g\nassert g.workers == 2 * multiprocessing.cpu_count() + 1, g.workers\nassert \"UvicornWorker\" in g.worker_class, g.worker_class\n", "La fórmula de workers es la esperada"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion33.md` a `auditorias/auditoria_sesion33.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
